# RTMOPose - Custom Training on Google Colab

Single-stage, anchor-free, real-time multi-person pose estimation.  
**Same dataset format and workflow as DETRPose** — just faster training.

| Model | Backbone | Params | ~Speed (A100) |
|-------|----------|--------|----------------|
| N     | HGNetV2-B0 nano  | ~4 M  | ~30 s/epoch |
| S     | HGNetV2-B0 small | ~8 M  | ~45 s/epoch |
| M     | HGNetV2-B2       | ~20 M | ~75 s/epoch |
| L     | HGNetV2-B4       | ~50 M | ~120 s/epoch |

## Workflow
1. **Settings** – fill in your Google Drive paths, model size, training options  
2. **Mount Drive** – authenticate Google Drive  
3. **Check GPU** – verify A100/T4 runtime  
4. **Initialize Environment** – clone repo, install dependencies (cached for repeat runs)  
5. **Fetch Dataset** – extract COCO-format data from Drive  
6. **Fetch Weights** – (optional) copy pretrained backbone weights  
7. **Train** – launch training with live output  
8. **Export ONNX** – export best checkpoint to ONNX  
9. **Export TensorRT** – (optional, Linux/Lambda.ai) convert ONNX to TRT engine  
10. **Inference** – run PyTorch, ONNX, or TRT inference  
11. **Save to Drive** – copy results to Google Drive

## Settings
Fill in the paths and options below before running any other cells.

In [ ]:

# ── Google Drive paths ────────────────────────────────────────────────────────
PROJECT_NAME = 'Dev Project/keypoints'   # Folder inside MyDrive/Datasets/
OBJECT_NAME  = 'Test_Coco/coco'          # Dataset folder name (contains train.7z, val.7z)

# ── Model ─────────────────────────────────────────────────────────────────────
# 'n' = nano  |  's' = small  |  'm' = medium  |  'l' = large
MODEL_SIZE = 's'

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS         = 200    # Override epoch count (set to None to use config default)
BATCH_SIZE     = 32     # Override total batch size (set to None to use config default)
NUM_WORKERS    = 8      # DataLoader worker processes per split
EVAL_INTERVAL  = 5      # Run COCO evaluation every N epochs (1 = every epoch)
USE_AMP        = True   # Mixed-precision training (fp16) — ~1.5-2x faster
BACKUP_EVERY_N_EPOCHS = 10   # Back up output to Google Drive every N epochs

# ── Resume training ───────────────────────────────────────────────────────────
RESUME_TRAINING   = False
RESUME_CHECKPOINT = 'checkpoint.pth'   # filename inside output dir on Google Drive

# ── Pretrained backbone weights (optional) ────────────────────────────────────
# Set to None to auto-download, or a GDrive filename inside weights/ folder
PRETRAIN_DFINE_WEIGHTS = None

# ── GitHub repo ────────────────────────────────────────────────────────────────
GITHUB_REPO_URL = 'https://github.com/GrayMitch/DETRPose-Remastered'
REPO_BRANCH     = 'main'

# ── Cached packages (speeds up environment setup on repeat runs) ──────────────
CACHED_RTMOPOSE_VERSION = '1.0.0'

# ─────────────────────────────────────────────────────────────────────────────
# Derived paths (do not edit below)
# ─────────────────────────────────────────────────────────────────────────────
import os

REPO_NAME          = 'DETRPose-Remastered'
REPO_LOCAL_PATH    = f'/content/{REPO_NAME}'
DATA_LOCAL_PATH    = '/content/coco_data'
OUTPUT_LOCAL_PATH  = os.path.join(REPO_LOCAL_PATH, 'output')

GDRIVE_DATASET_PATH   = f'/content/drive/MyDrive/Datasets/{PROJECT_NAME}/{OBJECT_NAME}'
GDRIVE_WEIGHTS_PATH   = os.path.join(GDRIVE_DATASET_PATH, 'weight')
GDRIVE_INSTALLER_PATH = '/content/drive/MyDrive/Installer/Colab'
GDRIVE_OUTPUT_PATH    = os.path.join(GDRIVE_DATASET_PATH, 'training_runs')
GDRIVE_BACKUP_PATH    = os.path.join(GDRIVE_DATASET_PATH, 'training_runs_backup_rtmo')

CONFIG_FILE = f'configs/rtmopose/rtmopose_hgnetv2_{MODEL_SIZE}_custom.py'
OUTPUT_DIR  = f'output/rtmopose_hgnetv2_{MODEL_SIZE}_custom'

VENV_PATH        = '/content/venv'
LOCAL_CACHE_PATH = '/content/cache'

CACHED_SITE_PACKAGES_ARCHIVE = f'site-packages_detrpose-{CACHED_RTMOPOSE_VERSION}.7z'
CACHED_SITE_PACKAGES_PATH    = os.path.join(GDRIVE_INSTALLER_PATH, CACHED_SITE_PACKAGES_ARCHIVE)

CUDA_VERSION      = '12.4.0'
CUDNN_VERSION     = '9.16.0'
CACHED_CUDA_ARCHIVE = f'cuda-{CUDA_VERSION}-cudnn-{CUDNN_VERSION}.7z'
CACHED_CUDA_PATH    = os.path.join(GDRIVE_INSTALLER_PATH, CACHED_CUDA_ARCHIVE)

PRETRAIN_LOCAL_PATH = None

print(f'Model size    : {MODEL_SIZE}')
print(f'Config file   : {CONFIG_FILE}')
print(f'Output dir    : {OUTPUT_DIR}')
print(f'Dataset       : {GDRIVE_DATASET_PATH}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Eval interval : every {EVAL_INTERVAL} epoch(s)')

## Mount Google Drive

In [ ]:
from google.colab import drive  # type: ignore
drive.mount('/content/drive')

## Check Colab Runtime Type

In [ ]:
import subprocess
print('=== Checking GPU ===')
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    )
    print(result.stdout.strip())
except Exception as e:
    print(f'nvidia-smi failed: {e}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## Initialize Environment
Clones the DETRPose-Remastered repo from GitHub, then installs all dependencies.  
On repeat runs a cached site-packages archive on Google Drive is used to skip pip install (~30 s vs ~5 min).

In [ ]:
import os
import shutil
import subprocess

os.makedirs(LOCAL_CACHE_PATH, exist_ok=True)

pip_path           = os.path.join(VENV_PATH, 'bin', 'pip')
python_path        = os.path.join(VENV_PATH, 'bin', 'python')
site_packages_path = os.path.join(VENV_PATH, 'lib', 'python3.10', 'site-packages')

# ── 1. Clone repo ────────────────────────────────────────────────────────────
print(f'=== Cloning {REPO_NAME} ===')
if os.path.exists(REPO_LOCAL_PATH):
    shutil.rmtree(REPO_LOCAL_PATH)
subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
     GITHUB_REPO_URL, REPO_LOCAL_PATH],
    check=True
)
print(f'Cloned to {REPO_LOCAL_PATH}')

# ── 2. Create virtual environment ────────────────────────────────────────────
if not os.path.exists(VENV_PATH):
    print('=== Creating virtual environment ===')
    subprocess.run(['python3', '-m', 'venv', VENV_PATH, '--system-site-packages'], check=True)

# ── 3. Install CUDA + cuDNN (cached or fresh) ────────────────────────────────
cuda_dir = os.path.join(VENV_PATH, 'cuda')
if not os.path.isdir(cuda_dir):
    if os.path.isfile(CACHED_CUDA_PATH):
        print('=== Extracting cached CUDA ===')
        subprocess.run(['7z', 'x', CACHED_CUDA_PATH, f'-o{VENV_PATH}', '-y'], check=True)
        print('CUDA extracted from cache')
    else:
        print('=== Installing CUDA + cuDNN via pip ===')
        subprocess.run([pip_path, 'install', '--upgrade',
                        f'nvidia-cuda-runtime-cu12=={CUDA_VERSION}',
                        f'nvidia-cudnn-cu12=={CUDNN_VERSION}',
                        '--target', cuda_dir], check=True)
        print('CUDA installed')
else:
    print('CUDA already available')

# ── 4. Install Python dependencies (cached or fresh) ────────────────────────
if os.path.isdir(site_packages_path) and len(os.listdir(site_packages_path)) > 10:
    print('Virtual env packages already present — skipping pip install')
elif os.path.isfile(CACHED_SITE_PACKAGES_PATH):
    print('=== Restoring site-packages from cache ===')
    subprocess.run(['7z', 'x', CACHED_SITE_PACKAGES_PATH, f'-o{site_packages_path}', '-y'], check=True)
    print('site-packages restored from cache')
else:
    print('=== Installing Python dependencies (first run — ~5 min) ===')
    req_file = os.path.join(REPO_LOCAL_PATH, 'requirements.txt')
    subprocess.run([pip_path, 'install', '-r', req_file], check=True)
    print('Dependencies installed')

print('Environment ready')

## Fetch Dataset from Google Drive
Copies `train.7z` and `val.7z` from Google Drive and extracts them into the expected COCO layout.

In [ ]:
import os
import shutil
import subprocess

if os.path.exists(DATA_LOCAL_PATH):
    shutil.rmtree(DATA_LOCAL_PATH)
os.makedirs(DATA_LOCAL_PATH, exist_ok=True)

for split in ['train', 'val']:
    src = os.path.join(GDRIVE_DATASET_PATH, f'{split}.7z')
    dst = os.path.join(DATA_LOCAL_PATH, f'{split}.7z')
    if not os.path.isfile(src):
        raise FileNotFoundError(f'Dataset archive not found: {src}')
    print(f'Copying {split}.7z ...')
    shutil.copy2(src, dst)
    print(f'Extracting {split}.7z ...')
    subprocess.run(['7z', 'x', dst, f'-o{DATA_LOCAL_PATH}', '-y'], check=True)
    os.remove(dst)

print('Dataset ready at', DATA_LOCAL_PATH)

## Fetch Pretrained Weights (Optional)
Copies backbone/DFine pretrain weights from Google Drive.  
Upload weights to `MyDrive/Datasets/{PROJECT_NAME}/{OBJECT_NAME}/weight/` and set `PRETRAIN_DFINE_WEIGHTS` in Settings.

In [ ]:
import os
import shutil

PRETRAIN_LOCAL_PATH = None

if PRETRAIN_DFINE_WEIGHTS:
    gdrive_weight = os.path.join(GDRIVE_WEIGHTS_PATH, PRETRAIN_DFINE_WEIGHTS)
    if not os.path.isfile(gdrive_weight):
        raise FileNotFoundError(f'Pretrain weights not found: {gdrive_weight}')
    local_weights_dir = os.path.join(REPO_LOCAL_PATH, 'weights', 'hgnetv2')
    os.makedirs(local_weights_dir, exist_ok=True)
    dst = os.path.join(local_weights_dir, PRETRAIN_DFINE_WEIGHTS)
    shutil.copy2(gdrive_weight, dst)
    PRETRAIN_LOCAL_PATH = dst
    print(f'Pretrain weights copied to: {dst}')
else:
    print('No pretrain weights specified — backbone will auto-download if needed.')

## (Optional) Re-fetch Repo
Run this cell to delete the locally cloned repo and re-clone it from GitHub.  
Useful if you pushed code changes and want to pick them up without restarting the runtime.

In [ ]:
import os
import shutil
import subprocess

if os.path.exists(REPO_LOCAL_PATH):
    shutil.rmtree(REPO_LOCAL_PATH)
    print(f'Deleted {REPO_LOCAL_PATH}')

subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
     GITHUB_REPO_URL, REPO_LOCAL_PATH],
    check=True
)
print(f'Re-cloned to {REPO_LOCAL_PATH}')

## Train Model
Runs `train.py` with the selected RTMOPose config.  
Output is streamed live and periodically backed up to Google Drive.

> **Tip:** RTMOPose is significantly faster to train than DETRPose (~10x fewer epochs needed for comparable accuracy).

In [ ]:
import datetime
import os
import shutil
import subprocess
from zoneinfo import ZoneInfo

python_path = os.path.join(VENV_PATH, 'bin', 'python')

# ── Build training command ────────────────────────────────────────────────────
cmd = [python_path, 'train.py', '--config_file', CONFIG_FILE]

options = [f'training_params.output_dir="{OUTPUT_DIR}"']
options.append(f'training_params.backup_every_n_epochs={BACKUP_EVERY_N_EPOCHS}')
options.append(f'training_params.gdrive_backup_path="{GDRIVE_BACKUP_PATH}"')

if EPOCHS is not None:
    options.append(f'training_params.epochs={EPOCHS}')
    lr_milestone = int(EPOCHS * 0.8)
    options.append(f'lr_scheduler.milestones=[{lr_milestone}]')

if BATCH_SIZE is not None:
    options.append(f'dataset_train.total_batch_size={BATCH_SIZE}')
    options.append(f'dataset_val.total_batch_size={BATCH_SIZE}')
    options.append(f'dataset_test.total_batch_size={BATCH_SIZE}')

if NUM_WORKERS is not None:
    options.append(f'dataset_train.num_workers={NUM_WORKERS}')
    options.append(f'dataset_val.num_workers={NUM_WORKERS}')
    options.append(f'dataset_test.num_workers={NUM_WORKERS}')

if EVAL_INTERVAL is not None:
    options.append(f'training_params.eval_interval={EVAL_INTERVAL}')

cmd += ['--options'] + options

if USE_AMP:
    cmd.append('--amp')

if RESUME_TRAINING:
    resume_ckpt = os.path.join(REPO_LOCAL_PATH, OUTPUT_DIR, RESUME_CHECKPOINT)
    if not os.path.isfile(resume_ckpt):
        gdrive_ckpt = os.path.join(GDRIVE_BACKUP_PATH, RESUME_CHECKPOINT)
        if os.path.isfile(gdrive_ckpt):
            os.makedirs(os.path.dirname(resume_ckpt), exist_ok=True)
            shutil.copy(gdrive_ckpt, resume_ckpt)
            print('Checkpoint copied from Google Drive backup')
        else:
            raise FileNotFoundError(
                f'Resume checkpoint not found:\n'
                f'  Local  : {resume_ckpt}\n'
                f'  GDrive : {gdrive_ckpt}'
            )
    cmd += ['--resume', resume_ckpt]
    print(f'Resuming from: {resume_ckpt}')

if PRETRAIN_LOCAL_PATH:
    cmd += ['--pretrain', PRETRAIN_LOCAL_PATH]

# ── Environment ───────────────────────────────────────────────────────────────
train_env = os.environ.copy()
train_env['CUDA_HOME']          = '/content/venv/cuda'
train_env['PATH']               = f"/content/venv/cuda/bin:{train_env.get('PATH', '')}"
train_env['LD_LIBRARY_PATH']    = f"/content/venv/cuda/lib64:{train_env.get('LD_LIBRARY_PATH', '')}"
train_env['CUDNN_PATH']         = '/content/venv/cuda'
train_env['RTMOPOSE_DATA_ROOT'] = DATA_LOCAL_PATH
train_env['MPLBACKEND']         = 'Agg'

# ── Run training ──────────────────────────────────────────────────────────────
tz = ZoneInfo('Africa/Johannesburg')
start_dt = datetime.datetime.now(datetime.timezone.utc).astimezone(tz)
print(f'Start time (+02:00): {start_dt.strftime("%Y/%m/%d %H:%M:%S")}')
print(f'Command: {" ".join(cmd)}')
print('=' * 80)

process = subprocess.Popen(
    cmd,
    cwd=REPO_LOCAL_PATH,
    env=train_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
returncode = process.returncode

end_dt = datetime.datetime.now(datetime.timezone.utc).astimezone(tz)
print('\n' + '=' * 80)
print(f'End time (+02:00): {end_dt.strftime("%Y/%m/%d %H:%M:%S")}')
print(f'Elapsed: {end_dt - start_dt}')
if returncode == 0:
    print('Training completed successfully!')
else:
    print(f'Training exited with code {returncode}')

## Export to ONNX
Exports the best checkpoint to an ONNX model compatible with the existing `onnx_inf.py` tool.

Output files:  
- `onnx_engines/rtmopose_hgnetv2_{size}_custom.onnx`  
- `onnx_engines/rtmopose_hgnetv2_{size}_custom_class_mappings.json`

In [ ]:
import os
import subprocess

python_path = os.path.join(VENV_PATH, 'bin', 'python')
config_abs  = os.path.join(REPO_LOCAL_PATH, CONFIG_FILE)
output_abs  = os.path.join(REPO_LOCAL_PATH, OUTPUT_DIR)
ckpt        = os.path.join(output_abs, 'checkpoint_best_regular.pth')

if not os.path.isfile(ckpt):
    ckpt = os.path.join(output_abs, 'checkpoint.pth')
    print(f'Best checkpoint not found, using latest: {ckpt}')

cmd = [
    python_path, 'tools/deployment/export_rtmo_onnx.py',
    '--config_file', config_abs,
    '--resume', ckpt,
]

train_env = os.environ.copy()
train_env['RTMOPOSE_DATA_ROOT'] = DATA_LOCAL_PATH

print(f'Exporting ONNX from: {ckpt}')
result = subprocess.run(cmd, cwd=REPO_LOCAL_PATH, env=train_env)
if result.returncode == 0:
    print('ONNX export successful!')
    onnx_out = os.path.join(REPO_LOCAL_PATH, 'onnx_engines')
    for f in sorted(os.listdir(onnx_out)):
        if 'rtmo' in f.lower():
            print(f'  {f}')
else:
    print(f'ONNX export failed (code {result.returncode})')

## Export to TensorRT (Linux / Lambda.ai only)
Converts the ONNX model to a TensorRT FP16 engine using `trtexec`.  
The existing `export_tensorrt.py` script auto-detects the RTMOPose ONNX file.

In [ ]:
import os
import subprocess

python_path = os.path.join(VENV_PATH, 'bin', 'python')

cmd = [python_path, 'tools/deployment/export_tensorrt.py', '--fp16']
result = subprocess.run(cmd, cwd=REPO_LOCAL_PATH)
if result.returncode == 0:
    print('TensorRT export successful!')
else:
    print(f'TensorRT export failed (code {result.returncode})')
    print('Ensure trtexec is at /usr/src/tensorrt/bin/trtexec')

## Run Inference — PyTorch
Runs inference on validation images using the trained RTMOPose PyTorch model.

In [ ]:
import os
import subprocess

python_path = os.path.join(VENV_PATH, 'bin', 'python')
ckpt        = os.path.join(REPO_LOCAL_PATH, OUTPUT_DIR, 'checkpoint_best_regular.pth')
source_dir  = os.path.join(DATA_LOCAL_PATH, 'val', 'images')
out_dir     = os.path.join(REPO_LOCAL_PATH, 'predictions', f'rtmo_{MODEL_SIZE}_pytorch')

cmd = [
    python_path, 'tools/inference/rtmo_inf.py',
    '--checkpoint', ckpt,
    '--config',     f'configs.rtmopose.rtmopose_hgnetv2_{MODEL_SIZE}_custom',
    '--source',     source_dir,
    '--output',     out_dir,
    '--conf',       '0.35',
]

train_env = os.environ.copy()
train_env['RTMOPOSE_DATA_ROOT'] = DATA_LOCAL_PATH

result = subprocess.run(cmd, cwd=REPO_LOCAL_PATH, env=train_env)
if result.returncode == 0:
    print(f'Predictions saved to {out_dir}')
else:
    print(f'Inference failed (code {result.returncode})')

## Run Inference — ONNX
Runs inference using the exported ONNX model.  
Uses the same `onnx_inf.py` tool as DETRPose — the output format is identical.

In [ ]:
import os
import subprocess

python_path = os.path.join(VENV_PATH, 'bin', 'python')
onnx_file   = os.path.join(REPO_LOCAL_PATH, 'onnx_engines',
                            f'rtmopose_hgnetv2_{MODEL_SIZE}_custom.onnx')
source_dir  = os.path.join(DATA_LOCAL_PATH, 'val', 'images')
out_dir     = os.path.join(REPO_LOCAL_PATH, 'predictions', f'rtmo_{MODEL_SIZE}_onnx')

cmd = [
    python_path, 'tools/inference/onnx_inf.py',
    '--onnx', onnx_file,
    '-i', source_dir,
    '-o', out_dir,
    '--conf', '0.35',
]

result = subprocess.run(cmd, cwd=REPO_LOCAL_PATH)
if result.returncode == 0:
    print(f'ONNX predictions saved to {out_dir}')
else:
    print(f'ONNX inference failed (code {result.returncode})')

## Run Inference — TensorRT
Runs inference using the TensorRT engine.

In [ ]:
import os
import subprocess

python_path = os.path.join(VENV_PATH, 'bin', 'python')
engine_file = os.path.join(REPO_LOCAL_PATH, 'trt_engines',
                            f'rtmopose_hgnetv2_{MODEL_SIZE}_custom.engine')
source_dir  = os.path.join(DATA_LOCAL_PATH, 'val', 'images')
out_dir     = os.path.join(REPO_LOCAL_PATH, 'predictions', f'rtmo_{MODEL_SIZE}_trt')

cmd = [
    python_path, 'tools/inference/trt_inf.py',
    '--engine', engine_file,
    '-i', source_dir,
    '-o', out_dir,
    '--conf', '0.35',
]

result = subprocess.run(cmd, cwd=REPO_LOCAL_PATH)
if result.returncode == 0:
    print(f'TRT predictions saved to {out_dir}')
else:
    print(f'TRT inference failed (code {result.returncode})')

## Copy Results to Google Drive
Copies the trained model outputs and ONNX/TRT engines to Google Drive.

In [ ]:
import os
import shutil

os.makedirs(GDRIVE_OUTPUT_PATH, exist_ok=True)

# Copy output checkpoints
local_out = os.path.join(REPO_LOCAL_PATH, OUTPUT_DIR)
if os.path.isdir(local_out):
    for f in os.listdir(local_out):
        if f.endswith('.pth') or f.endswith('.json'):
            src = os.path.join(local_out, f)
            dst = os.path.join(GDRIVE_OUTPUT_PATH, f)
            shutil.copy2(src, dst)
            print(f'  Copied checkpoint: {f}')

# Copy ONNX engines
onnx_local  = os.path.join(REPO_LOCAL_PATH, 'onnx_engines')
gdrive_onnx = os.path.join(GDRIVE_OUTPUT_PATH, 'onnx_engines')
os.makedirs(gdrive_onnx, exist_ok=True)
if os.path.isdir(onnx_local):
    for f in os.listdir(onnx_local):
        if 'rtmo' in f.lower():
            shutil.copy2(os.path.join(onnx_local, f), os.path.join(gdrive_onnx, f))
            print(f'  Copied ONNX: {f}')

print(f'Results copied to {GDRIVE_OUTPUT_PATH}')

## (Dev Only) Archive Packages for Cached Retrieval
Run this once after a successful fresh install to create a cached archive on Google Drive.  
Subsequent runs will restore from this archive instead of pip-installing (~30 s vs ~5 min).

In [ ]:
import os
import subprocess

os.makedirs(GDRIVE_INSTALLER_PATH, exist_ok=True)

def _archive_to_gdrive(src_path, gdrive_archive_path):
    print(f'Archiving {src_path} -> {gdrive_archive_path}')
    subprocess.run(
        ['7z', 'a', '-t7z', '-mx=3', gdrive_archive_path, '.'],
        cwd=src_path,
        check=True
    )
    print(f'Archived to {gdrive_archive_path}')

site_packages_path = os.path.join(VENV_PATH, 'lib', 'python3.10', 'site-packages')
_archive_to_gdrive(site_packages_path, CACHED_SITE_PACKAGES_PATH)

## RTMOPose Settings
Configure all training options here before running any other cells.

In [ ]:

PROJECT_NAME = 'Dev Project/keypoints'
OBJECT_NAME  = 'Test_Coco/coco'
MODEL_SIZE = 's'
EPOCHS = 200
BATCH_SIZE = 32
NUM_WORKERS = 8
EVAL_INTERVAL = 5
USE_AMP = True
BACKUP_EVERY_N_EPOCHS = 10
RESUME_TRAINING = False
RESUME_CHECKPOINT = 'checkpoint.pth'
PRETRAIN_DFINE_WEIGHTS = None
GITHUB_REPO_URL = 'https://github.com/GrayMitch/DETRPose-Remastered'
REPO_BRANCH = 'main'
CACHED_RTMOPOSE_VERSION = '1.0.0'
import os
REPO_NAME = 'DETRPose-Remastered'
REPO_LOCAL_PATH = f'/content/{REPO_NAME}'
DATA_LOCAL_PATH = '/content/coco_data'
GDRIVE_DATASET_PATH = f'/content/drive/MyDrive/Datasets/{PROJECT_NAME}/{OBJECT_NAME}'
GDRIVE_WEIGHTS_PATH = os.path.join(GDRIVE_DATASET_PATH, 'weight')
GDRIVE_INSTALLER_PATH = '/content/drive/MyDrive/Installer/Colab'
GDRIVE_OUTPUT_PATH = os.path.join(GDRIVE_DATASET_PATH, 'training_runs')
GDRIVE_BACKUP_PATH = os.path.join(GDRIVE_DATASET_PATH, 'training_runs_backup_rtmo')
CONFIG_FILE = f'configs/rtmopose/rtmopose_hgnetv2_{MODEL_SIZE}_custom.py'
OUTPUT_DIR = f'output/rtmopose_hgnetv2_{MODEL_SIZE}_custom'
VENV_PATH = '/content/venv'
LOCAL_CACHE_PATH = '/content/cache'
CACHED_SITE_PACKAGES_ARCHIVE = f'site-packages_detrpose-{CACHED_RTMOPOSE_VERSION}.7z'
CACHED_SITE_PACKAGES_PATH = os.path.join(GDRIVE_INSTALLER_PATH, CACHED_SITE_PACKAGES_ARCHIVE)
CUDA_VERSION = '12.4.0'
CUDNN_VERSION = '9.16.0'
CACHED_CUDA_ARCHIVE = f'cuda-{CUDA_VERSION}-cudnn-{CUDNN_VERSION}.7z'
CACHED_CUDA_PATH = os.path.join(GDRIVE_INSTALLER_PATH, CACHED_CUDA_ARCHIVE)
PRETRAIN_LOCAL_PATH = None
print(f'Model size : {MODEL_SIZE}')
print(f'Config file: {CONFIG_FILE}')
print(f'Output dir : {OUTPUT_DIR}')


## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')